## Carga/creacion de datos

In [2]:
# ============================================
# CASO: PREDICCIÓN DE DIABETES TIPO 2
# FLUJO COMPLETO DE DATA SCIENCE
# ============================================

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. GENERACIÓN DE DATASET SINTÉTICO
# ============================================

def generar_dataset_diabetes(n_muestras=5000, semilla=42):
    """
    Genera un dataset sintético de diabetes con características realistas
    """
    np.random.seed(semilla)
    
    # Características demográficas
    edad = np.random.normal(50, 15, n_muestras).clip(18, 90).astype(int)
    sexo = np.random.choice(['M', 'F'], n_muestras, p=[0.48, 0.52])
    
    # Características clínicas
    imc = np.random.normal(28, 6, n_muestras).clip(15, 50)
    presion_sistolica = np.random.normal(130, 15, n_muestras).clip(90, 200)
    presion_diastolica = np.random.normal(85, 10, n_muestras).clip(60, 120)
    glucosa = np.random.normal(100, 25, n_muestras).clip(70, 300)
    insulina = np.random.normal(85, 35, n_muestras).clip(15, 300)
    hba1c = np.random.normal(6.7, 1.2, n_muestras).clip(4.0, 14.0)
    
    # Estilo de vida y antecedentes
    actividad_fisica = np.random.choice(['Baja', 'Moderada', 'Alta'], n_muestras, p=[0.4, 0.35, 0.25])
    fuma = np.random.choice(['Si', 'No'], n_muestras, p=[0.3, 0.7])
    antecedentes_familiares = np.random.choice(['Si', 'No'], n_muestras, p=[0.35, 0.65])
    
    # Crear DataFrame
    df = pd.DataFrame({
        'Edad': edad,
        'Sexo': sexo,
        'IMC': imc.round(1),
        'Presion_Sistolica': presion_sistolica.round(0).astype(int),
        'Presion_Diastolica': presion_diastolica.round(0).astype(int),
        'Glucosa': glucosa.round(0).astype(int),
        'Insulina': insulina.round(1),
        'HbA1c': hba1c.round(1),
        'Actividad_Fisica': actividad_fisica,
        'Fuma': fuma,
        'Antecedentes_Familiares': antecedentes_familiares
    })
    
    # Función para determinar diabetes basada en factores de riesgo
    def determinar_diabetes(row):
        # Puntaje de riesgo
        riesgo = 0
        
        # Factores de riesgo ponderados
        if row['HbA1c'] >= 6.5:
            riesgo += 4
        elif row['HbA1c'] >= 5.7:
            riesgo += 2
            
        if row['Glucosa'] >= 126:
            riesgo += 3
        elif row['Glucosa'] >= 100:
            riesgo += 1
            
        if row['IMC'] >= 30:
            riesgo += 2
        elif row['IMC'] >= 25:
            riesgo += 1
            
        if row['Edad'] >= 45:
            riesgo += 1
            
        if row['Antecedentes_Familiares'] == 'Si':
            riesgo += 1
            
        if row['Actividad_Fisica'] == 'Baja':
            riesgo += 1
            
        # Probabilidad basada en el riesgo (con algo de ruido)
        prob_diabetes = 1 / (1 + np.exp(-(riesgo - 5)))
        return np.random.binomial(1, prob_diabetes)
    
    # Aplicar la función para generar la variable objetivo
    df['Diabetes'] = df.apply(determinar_diabetes, axis=1)
    
    # Ajustar para tener un balance aproximado (30% con diabetes)
    current_ratio = df['Diabetes'].mean()
    if current_ratio < 0.28 or current_ratio > 0.32:
        # Rebalancear ligeramente
        target_ratio = 0.30
        df_pos = df[df['Diabetes'] == 1]
        df_neg = df[df['Diabetes'] == 0]
        
        n_pos = int(n_muestras * target_ratio)
        n_neg = n_muestras - n_pos
        
        df_pos_sample = df_pos.sample(n=min(n_pos, len(df_pos)), replace=True, random_state=semilla)
        df_neg_sample = df_neg.sample(n=n_neg, replace=True, random_state=semilla)
        
        df = pd.concat([df_pos_sample, df_neg_sample]).sample(frac=1, random_state=semilla)
    
    return df

# Generar el dataset
df = generar_dataset_diabetes(n_muestras=5000, semilla=42)
print("Dataset generado exitosamente!")
print(f"Dimensiones: {df.shape}")
print(f"Distribución de Diabetes: {df['Diabetes'].value_counts(normalize=True).round(3)}")

# guardar df en un csv 'diabetes_sintetico.csv' en la carpeta 'data'
df.to_csv('../data/diabetes_sintetico.csv', index=False)

Dataset generado exitosamente!
Dimensiones: (5000, 12)
Distribución de Diabetes: Diabetes
0    0.7
1    0.3
Name: proportion, dtype: float64
